<a href="https://colab.research.google.com/github/rakeshma-72/LLM/blob/main/Experiment_6_LLM_Evaluation_and_Benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import pandas as pd
import math

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# Step 2: Select two pretrained language models
# DistilGPT2 is smaller; GPT-2 is the second model for comparison.
model_names = {    "DistilGPT2": "distilgpt2",    "GPT2": "gpt2" }

In [4]:
# Step 3: Prepare sample prompts and reference answers
 # Reference answers help us evaluate the models on the same text.
test_data = [
      {
          "prompt": "What is artificial intelligence?",
          "reference": "Artificial intelligence enables machines to perform tasks that normally require human intelligence."
          },
          {
              "prompt": "Why is exercise important?",
              "reference": "Exercise improves physical fitness and supports overall health."
          },
      {
          "prompt": "Explain the importance of education.",
          "reference": "Education develops knowledge, skills, and critical thinking."
      }
]

In [5]:
# Step 4: Create a list to store evaluation results
results = []

In [6]:
# Step 5: Load each model and evaluate it
for model_name, model_path in model_names.items():
   print("\\nEvaluating model:", model_name)
   tokenizer = AutoTokenizer.from_pretrained(model_path)
   model = AutoModelForCausalLM.from_pretrained(model_path)
   model.eval()
   device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
   model.to(device)
   tokenizer.pad_token = tokenizer.eos_token
   perplexities = []
   for item in test_data:
     prompt = item["prompt"]
     reference = item["reference"]
     inputs = tokenizer(prompt, return_tensors="pt").to(device)
     with torch.no_grad():
            output_ids = model.generate(
          **inputs,
          max_new_tokens=50,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id
          )
     generated_text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
        )
     reference_inputs = tokenizer(
          reference,
          return_tensors="pt"
          ).to(device)
     with torch.no_grad():
        outputs = model(
            input_ids=reference_inputs["input_ids"],
            labels=reference_inputs["input_ids"]
        )
     loss = outputs.loss.item()
     perplexity = math.exp(loss)
     perplexities.append(perplexity)
     text_lower = generated_text.lower()
     only_prompt = generated_text.strip() == prompt.strip()
     bias_terms = ["all women", "all men", "all people from"]
     possible_bias = any(term in text_lower for term in bias_terms)
     error_terms = ["i don't know", "cannot answer"]
     possible_issue = any(term in text_lower for term in error_terms)
     print("\\nPrompt:", prompt)
     print("Generated Output:", generated_text)
     print("Reference Answer:", reference)
     print("Perplexity:", round(perplexity, 2))
     print("Possible Bias:", possible_bias)
     print("Possible Issue:", possible_issue)

     results.append({
           "Model": model_name,
           "Prompt": prompt,
           "Generated Output": generated_text,
           "Reference Answer": reference,
           "Perplexity": round(perplexity, 2),
           "Possible Bias": possible_bias,
           "Possible Issue": possible_issue,
           "Only Prompt": only_prompt

       })
   del model
   if torch.cuda.is_available():
       torch.cuda.empty_cache()

   avg_perplexity = sum(perplexities) / len(perplexities)
   print("\\nAverage Perplexity:", round(avg_perplexity, 2))


df = pd.DataFrame(results)
print(df[["Model", "Prompt", "Perplexity"]].to_string(index=False))
df.to_csv("llm_evaluation_results.csv", index=False)

print("\\nEvaluation completed!")
print("Results saved to llm_evaluation_results.csv")



\nEvaluating model: DistilGPT2


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (12 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

\nPrompt: What is artificial intelligence?
Generated Output: What is artificial intelligence?

AI is a new field of research that has been around for a long time. It is a new field of research that has been around for a long time. It is a new field of research that has been around for a long time.
Reference Answer: Artificial intelligence enables machines to perform tasks that normally require human intelligence.
Perplexity: 24.9
Possible Bias: False
Possible Issue: False
\nPrompt: Why is exercise important?
Generated Output: Why is exercise important?

The answer is that exercise is important. It's important to get your body moving, to get your mind moving, to get your body moving. It's important to get your mind moving.

The key to getting your mind moving is
Reference Answer: Exercise improves physical fitness and supports overall health.
Perplexity: 76.26
Possible Bias: False
Possible Issue: False
\nPrompt: Explain the importance of education.
Generated Output: Explain the importan